In [7]:
import pandas as pd
import glob
import os

from utils import *

In [ ]:
DATA_DIR = '../00_data/raw/'
DATA_SAVEDIR = '../00_data/processed/'
organism_name = "e_coli"
# organism_name = "s_cerevisiae"
# organism_name = "b_subtilis"
# organism_name = "a_thaliana"
# organism_name = 'm_musculus'
# organism_name = 'p_marinus'
# organism_name = 'h_sapiens'
# organism_name = 'g_sulphuraria'
# organism_name = 'r_capsulatus'
# organism_name = 'r_gelatinosus'
# organism_name = 'p_fluorescens'
# organism_name = 'p_citronellolis'
# organism_name = 'p_stutzerii'
# organism_name = 'p_protegens'
# organism_name = 'p_rubens'
# organism_name = 'p_pantotrophus'

CPDS_FILE = glob.glob(os.path.join(DATA_DIR, organism_name,"*compounds*"))[0]
REACTIONS_FILE = glob.glob(os.path.join(DATA_DIR, organism_name,"*reactions*"))[0]

SAVEDIR = os.path.join(DATA_SAVEDIR, organism_name)
if not os.path.exists(SAVEDIR):
    os.mkdir(SAVEDIR)
SAVE_NAME = f"{SAVEDIR}/{organism_name}_metabolites_from_pathways.csv"

print (CPDS_FILE)
print (REACTIONS_FILE)
print (SAVE_NAME)

reaction_df = pd.read_csv(REACTIONS_FILE, sep='\t').dropna(subset=['In-Pathway'])
cpds_df = pd.read_csv(CPDS_FILE, sep='\t').dropna()
cpds_to_smiles = pd.Series(cpds_df['SMILES'].values, index=cpds_df['Compound'].values).to_dict()

metabolites_strings = set([])
for x in reaction_df['Substrates']:
    for met in [y.strip() for y in x.split(' // ')]:
        metabolites_strings.add(met)
        
metabolite_names = []
metabolite_smiles = []
for s in metabolites_strings:
    try:
        metabolite_smiles.append(standardize_smiles(cpds_to_smiles[s]))
        metabolite_names.append(s)
    except:
        print ('Could not convert {} to SMILES'.format(s))
        pass

print ("Number of metabolites:", len(metabolite_smiles))    

In [ ]:
## Uncommment to save

pd.DataFrame({'name':metabolite_names, 'smiles':metabolite_smiles}).to_csv(SAVE_NAME, index=False, sep='\t')
print ("DATA SAVED TO", SAVE_NAME)